### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="rossmann_store_sales",
    dataset_year="2015",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/rossmann-store-sales/overview",
    download_description=r"""
kaggle competitions download -c rossmann-store-sales \
&& mkdir -p local-data-warehouse/rossmann_store_sales \
&& mv rossmann-store-sales.zip local-data-warehouse/rossmann_store_sales \
&& unzip local-data-warehouse/rossmann_store_sales/rossmann-store-sales.zip -d local-data-warehouse/rossmann_store_sales \
&& rm local-data-warehouse/rossmann_store_sales/rossmann-store-sales.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{kaggle_rossmann_store_sales,
    title        = {Rossmann Store Sales},
    author       = {{Kaggle}},
    howpublished = {\url{https://www.kaggle.com/competitions/rossmann-store-sales/overview}},
    note         = {Kaggle competition page. Accessed: 2026-03-19},
    year         = {2015}
}
""",
    academic_reference_bibtex_key="kaggle_rossmann_store_sales",
    license="Kaggle License",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
- We merge the store data with the train data.
- The data is from a Kaggle competition with a temporal split, so the test data cannot be used for unsupervised approachs.
- The winning solution is public: https://storage.googleapis.com/kaggle-forum-message-attachments/102102/3454/Rossmann_nr1_doc.pdf.
- The competition used 48 days for testing. The description says "Rossmann store managers are tasked with predicting their daily sales for up to six weeks in advance". Therefore, we define the same horizon for defining our test splits and add one day between train/test as a planning gap.
- We transform the date column to datetime format.
- We exclude days when the store is closed since they all have zero sales and are trivial to predict. We drop the Open column.
- We assign categorical data type to the Store column and the DayOfWeek column.
- For the train data, the # of customers per day is given, which is not available at the prediction point for the test data. The 1st-place solution says that one model variation computed recent-history features on the number of customers instead of sales, and also used store-level aggregates such as average sales per customer. However this would mean that we have to define split-specific splits. Since many high performing solutions did not utilize customer information, we drop the column.
- The StateHoliday column has two encodings for the entry 0, one as number and one as object. We combine them into one category.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Sales",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="Date",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

'''
NOTES
- We need to do some FE to make the "Customers" column useful. Test how much the full aggregated set differs from the splits.
'''
train = pd.read_csv(dataset_mold.path / "train.csv")
store = pd.read_csv(dataset_mold.path / "store.csv")
test = pd.read_csv(dataset_mold.path / "test.csv")

df = pd.merge(train, store, on="Store", how="left")
test = pd.merge(test, store, on="Store", how="left")

df["Date"] = pd.to_datetime(df["Date"])
test["Date"] = pd.to_datetime(test["Date"])

df = df[df["Open"] == 1].copy()
test = test[test["Open"] == 1].copy()

df = df.drop(columns=["Open"])
test = test.drop(columns=["Open"])

for col in ["Store", "DayOfWeek"]:
    df[col] = df[col].astype("category")
    test[col] = test[col].astype(df[col].dtype)

# Drop customer information. TODO: Enable split-specific data storage and feature engineering to utilize this information.
df = df.drop(columns=["Customers"])

df["StateHoliday"] = df["StateHoliday"].astype(str).astype("category")

as_cat_dtype = ["StoreType", "Assortment", "PromoInterval"]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")

print("Loaded data shape:", df.shape)
df = df.sort_values("Date").reset_index(drop=True)

/tmp/ipykernel_178082/4068176971.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(dataset_mold.path / "train.csv")


Loaded data shape: (844392, 16)


In [3]:
df.Date.describe(),test.Date.describe()

(count                           844392
 mean     2014-04-11 01:02:42.487565312
 min                2013-01-01 00:00:00
 25%                2013-08-16 00:00:00
 50%                2014-03-31 00:00:00
 75%                2014-12-10 00:00:00
 max                2015-07-31 00:00:00
 Name: Date, dtype: object,
 count                            35093
 mean     2015-08-24 18:39:34.011911168
 min                2015-08-01 00:00:00
 25%                2015-08-13 00:00:00
 50%                2015-08-25 00:00:00
 75%                2015-09-05 00:00:00
 max                2015-09-17 00:00:00
 Name: Date, dtype: object)

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 844,392
Columns: 16
Use sampling: False (sample size: 844,392)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Store', 'Date', 'CompetitionDistance', 'Promo2SinceWeek', 'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'DayOfWeek', 'Promo2SinceYear', 'StoreType', 'StateHoliday']
Rows remaining as candidates after top-10 filter: 0 (of 844,392)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,Store,DayOfWeek,Date,Sales,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1097,2,2013-01-01,5961,0,a,1,b,b,720.0,3.0,2002.0,0,NaN,NaN,NaN
1,85,2,2013-01-01,4220,0,a,1,b,a,1870.0,10.0,2011.0,0,NaN,NaN,NaN
2,259,2,2013-01-01,6851,0,a,1,b,b,210.0,NaN,NaN,0,NaN,NaN,NaN
3,262,2,2013-01-01,17267,0,a,1,b,a,1180.0,5.0,2013.0,0,NaN,NaN,NaN
4,274,2,2013-01-01,3102,0,a,1,b,b,3640.0,NaN,NaN,1,10.0,2013.0,"Jan,Apr,Jul,Oct"


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,PromoInterval,category,423307.0,50.13,3.0,"Jan,Apr,Jul,Oct, Feb,May,Aug,Nov, Mar,Jun,Sept,Dec"
1,Store,category,0.0,0.00,1115.0,"1097, 423, 769, 682, 494, 562, 335, 85, 733, 262"
2,DayOfWeek,category,0.0,0.00,7.0,"6, 2, 3, 5, 1, 4, 7"
3,StateHoliday,category,0.0,0.00,4.0,"0, a, b, c"
4,StoreType,category,0.0,0.00,4.0,"a, d, c, b"
5,Assortment,category,0.0,0.00,3.0,"a, c, b"
6,Date,datetime64[ns],0.0,0.00,942.0,"2015-06-10 00:00:00, 2015-06-30 00:00:00, 2015-06-09 00:00:00, 2015-06-08 00:00:00, 2015-06-05 00:00:00, 2015-06-02 00:00:00, 2015-06-01 00:00:00, 2015-05-30 00:00:00, 2014-05-05 00:00:00, 2014-05-06 00:00:00"
7,Promo2SinceWeek,float64,423307.0,50.13,24.0,"14.0, 40.0, 31.0, 10.0, 5.0, 37.0, 1.0, 13.0, 45.0, 22.0"
8,Promo2SinceYear,float64,423307.0,50.13,7.0,"2011.0, 2013.0, 2014.0, 2012.0, 2009.0, 2010.0, 2015.0"
9,CompetitionOpenSinceMonth,float64,268619.0,31.81,12.0,"9.0, 4.0, 11.0, 3.0, 7.0, 12.0, 10.0, 6.0, 5.0, 2.0"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Sales,844392.0,6955.514291,3104.214680,0.0,41551.0
Promo,844392.0,0.446352,0.497114,0.0,1.0
SchoolHoliday,844392.0,0.193580,0.395103,0.0,1.0
CompetitionDistance,842206.0,5457.979627,7809.437311,20.0,75860.0
CompetitionOpenSinceMonth,575773.0,7.224879,3.210144,1.0,12.0
CompetitionOpenSinceYear,575773.0,2008.697747,5.978048,1900.0,2015.0
Promo2,844392.0,0.498684,0.499999,0.0,1.0
Promo2SinceWeek,421085.0,23.253426,14.100569,1.0,50.0
Promo2SinceYear,421085.0,2011.754019,1.660962,2009.0,2015.0


In [8]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column        rank                                    
Assortment    1                       a  444909  52.69
              2                       c  391271  46.34
              3                       b    8212   0.97
Date          1     2015-06-10 00:00:00    1115   0.13
              2     2015-06-30 00:00:00    1115   0.13
              3     2015-06-09 00:00:00    1115   0.13
              4     2015-06-08 00:00:00    1115   0.13
              5     2015-06-05 00:00:00    1115   0.13
DayOfWeek     1                       6  144058  17.06
              2                       2  143961  17.05
              3                       3  141936  16.81
              4                       5  138640  16.42
              5                       1  137560  16.29
PromoInterval 1                    <NA>  423307  50.13
              2         Jan,Apr,Jul,Oct  242411  28.71
              3         Feb,May,Aug,Nov   98005  11.61
              4        Mar,Jun,Sept,Dec   80669   9.55
StateHoliday  1                       0  843482  99.89
              2                       a     694   0.08
              3                       b     145   0.02
              4                       c      71   0.01
Store         1                    1097     942   0.11
              2                     423     942   0.11
              3                     769     942   0.11
              4                     682     942   0.11
              5                     494     942   0.11
StoreType     1                       a  457077  54.13
              2                       d  258774  30.65
              3                       c  112978  13.38
              4                       b   15563   1.84

In [9]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.01,1.594,-0.638,9636148.782,0.186,log1p,16629029.6,15820859.0,lognormal


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=1, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

splits = {}
used_in_train = set()
used_in_test = set()
used_data = set()

last_date = df[date_col].max()

test_length = 42
planning_gap = 1

for split in range(n_splits * n_repeats):
    # walk backwards in 42-day blocks
    test_end_day = last_date - pd.DateOffset(days=split * test_length)
    test_start_day = test_end_day - pd.DateOffset(days=test_length - 1)

    # 1-day planning gap before test starts
    pred_point = test_start_day - pd.DateOffset(days=planning_gap)

    train_idx = df.loc[df[date_col] < pred_point].index
    test_idx = df.loc[
        (df[date_col] >= test_start_day) &
        (df[date_col] <= test_end_day)
    ].index

    assert len(set(train_idx).intersection(set(test_idx))) == 0, "Train and test should not overlap"

    if split > 0:
        prev_test_idx = splits[split - 1][0][1]
        assert len(set(test_idx).intersection(set(prev_test_idx))) == 0, "Test splits overlap"

    # if you expect daily data with no missing dates:
    test_dates = df.loc[test_idx, date_col]
    assert (test_dates.max() - test_dates.min()).days == test_length - 1, "Test window should span 42 days"

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Test start:", test_start_day, "| Test end:", test_end_day)
    print("Train target mean:", df.loc[train_idx, target_col].mean())
    print("Test target mean:", df.loc[test_idx, target_col].mean())

    splits[split] = {0: [train_idx, test_idx]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata( 
    splits_comment=f"The description says 'Rossmann store managers are tasked with predicting their daily sales for up to six weeks in advance'. \
        Therefore, we define the same horizon for our test splits and add one day between train/test as a planning gap.",
    splits=splits
)


=== Step 0 ===
Train size: 802996 | Test size: 40282
Test start: 2015-06-20 00:00:00 | Test end: 2015-07-31 00:00:00
Train target mean: 6953.470344808691
Test target mean: 6978.638051735266

=== Step 1 ===
Train size: 765523 | Test size: 37473
Test start: 2015-05-09 00:00:00 | Test end: 2015-06-19 00:00:00
Train target mean: 6937.202652304372
Test target mean: 7280.26651188856

=== Step 2 ===
Train size: 728478 | Test size: 37044
Test start: 2015-03-28 00:00:00 | Test end: 2015-05-08 00:00:00
Train target mean: 6903.028150472629
Test target mean: 7652.293164885002
1.0000 of the samples are used.
0.9510 of the samples are used in training
0.1360 of the samples are used in testing.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to rossmann_store_sales/019db4fb-6d9e-7c15-b8cb-1bf97ea88782
019db4fb-6d9e-7c15-b8cb-1bf97ea88782
79bb9fde7a4e4a5c16a8bd5f5171ad538fb97c66529695b71b21051d058d30be
